In [1]:
#!pip install torchvision
#!pip install facenet_pytorch

import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
# from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
# from vivit import ViViTBackbone
import gc
from VideoSwinTransformer import SwinTransformer3D
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="r2plus1d+VST + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()


In [2]:

logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:2' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-20 21:19:43,643]::2853905153::DEBUG::=============r2plus1d+VST + AST fusion after transformer blocks+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:2


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:
best_model_path = './save_swintransformer_folder/model_best_multiseed.pth'
os.makedirs(os.path.dirname(best_model_path), exist_ok=True)
print(1)


1


In [5]:
validation_set_data=getdata("/data1/qx/mean_based_arg/KETI/data/scene3_valid_set30s.dat","valid")
print(1)
train_set_data=getdata("/data1/qx/mean_based_arg/KETI/data/scene3_train_set30s.dat","train")
print(1)

1
1


In [6]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [7]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)/120
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [8]:
num_workerssz = 20
lr = 3e-05
epochs = 120
print(1)

1


In [9]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0], fullshot=train_set_data[1], tagdata=train_set_data[3], transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0], fullshot=validation_set_data[1], tagdata=validation_set_data[3], transform=transform)
print(1)


1


In [10]:
criterion = torch.nn.L1Loss().to(device)
print(1)


1


In [11]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [12]:
SEEDS = [42, 123, 777, 1004, 2024]

patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


seed_results = []
global_best_mae = float('inf')
global_best_seed = None
global_best_epoch = None

with torch.cuda.device(2):

    for seed_idx, seed in enumerate(SEEDS, start=1):

        logger.info(
            '================ Seed {}/{}: {} ================'.format(
                seed_idx, len(SEEDS), seed
            )
        )

        set_seed(seed)

        # ----------------------------------------------------
        # Rebuild DataLoaders for every seed
        # ----------------------------------------------------
        train_generator = torch.Generator()
        train_generator.manual_seed(seed)

        val_generator = torch.Generator()
        val_generator.manual_seed(seed + 10000)

        train_dataloader = DataLoader(
            dataset=train_set_data,
            batch_size=batchsz,
            shuffle=True,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=train_generator
        )

        val_dataloader = DataLoader(
            dataset=val_set_data,
            batch_size=batchsz,
            shuffle=False,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=val_generator
        )

        # ----------------------------------------------------
        # Fresh model / optimizer for every seed
        # ----------------------------------------------------
        model = SwinTransformer3D()
        model.to(device)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr
        )

        seed_best_mae = float('inf')
        seed_best_epoch = 0
        no_improve_count = 0

        for epoch in range(epochs):

            # ==================================================
            # Train
            # ==================================================
            model.train()

            train_loss_sum = 0.0

            for audiodata, fullshot, big_five_data in tqdm(
                train_dataloader,
                desc='Seed {} Train Epoch {}'.format(seed, epoch + 1)
            ):
                fullshot = fullshot.permute(0, 4, 1, 2, 3)
                big_five_data = big_five_data.permute(0, 2, 1).squeeze()

                audiodata = audiodata.to(device)
                fullshot = fullshot.to(device)
                big_five_data = big_five_data.to(device)

                optimizer.zero_grad()

                hypothesis = model(
                    fullshot,
                    audiodata
                )

                loss = criterion(
                    hypothesis,
                    big_five_data
                )

                loss.backward()
                optimizer.step()

                train_loss_sum += loss.item()

            train_avg_loss = train_loss_sum / len(train_dataloader)

            # ==================================================
            # Validation
            # ==================================================
            model.eval()

            val_loss_sum = 0.0
            val_o_sum = 0.0
            val_c_sum = 0.0
            val_e_sum = 0.0
            val_a_sum = 0.0
            val_n_sum = 0.0

            with torch.no_grad():

                for audiodata, fullshot, big_five_data in tqdm(
                    val_dataloader,
                    desc='Seed {} Val Epoch {}'.format(seed, epoch + 1)
                ):
                    fullshot = fullshot.permute(0, 4, 1, 2, 3)
                    big_five_data = big_five_data.permute(0, 2, 1).squeeze()

                    audiodata = audiodata.to(device)
                    fullshot = fullshot.to(device)
                    big_five_data = big_five_data.to(device)

                    hypothesis = model(
                        fullshot,
                        audiodata
                    )

                    val_loss = criterion(
                        hypothesis,
                        big_five_data
                    )

                    (
                        hypothesiso,
                        hypothesisc,
                        hypothesise,
                        hypothesisa,
                        hypothesisn
                    ) = extract_ocean(
                        hypothesis,
                        len(fullshot)
                    )

                    (
                        big_five_datao,
                        big_five_datac,
                        big_five_datae,
                        big_five_dataa,
                        big_five_datan
                    ) = extract_ocean(
                        big_five_data,
                        len(fullshot)
                    )

                    val_o_loss = criterion(hypothesiso, big_five_datao)
                    val_c_loss = criterion(hypothesisc, big_five_datac)
                    val_e_loss = criterion(hypothesise, big_five_datae)
                    val_a_loss = criterion(hypothesisa, big_five_dataa)
                    val_n_loss = criterion(hypothesisn, big_five_datan)

                    val_loss_sum += val_loss.item()
                    val_o_sum += val_o_loss.item()
                    val_c_sum += val_c_loss.item()
                    val_e_sum += val_e_loss.item()
                    val_a_sum += val_a_loss.item()
                    val_n_sum += val_n_loss.item()

            val_avg_loss = val_loss_sum / len(val_dataloader)
            val_avg_o_loss = val_o_sum / len(val_dataloader)
            val_avg_c_loss = val_c_sum / len(val_dataloader)
            val_avg_e_loss = val_e_sum / len(val_dataloader)
            val_avg_a_loss = val_a_sum / len(val_dataloader)
            val_avg_n_loss = val_n_sum / len(val_dataloader)

            # ==================================================
            # Per-seed best + patience
            # ==================================================
            if val_avg_loss < seed_best_mae:

                seed_best_mae = val_avg_loss
                seed_best_epoch = epoch + 1
                no_improve_count = 0

                # Save ONLY when this is also the best model
                # across all seeds. The same file is overwritten,
                # therefore only one checkpoint remains on disk.
                if val_avg_loss < global_best_mae:

                    global_best_mae = val_avg_loss
                    global_best_seed = seed
                    global_best_epoch = epoch + 1

                    torch.save(
                        {
                            'seed': seed,
                            'epoch': epoch + 1,
                            'model': model.state_dict(),
                            'optimizer': optimizer.state_dict(),
                            'val_mae': val_avg_loss,
                            'val_1_mae': 1.0 - val_avg_loss
                        },
                        best_model_path
                    )

                    logger.info(
                        'Global best checkpoint updated: '
                        'seed={}, epoch={}, MAE={:.6f}, 1-MAE={:.6f}'.format(
                            seed,
                            epoch + 1,
                            val_avg_loss,
                            1.0 - val_avg_loss
                        )
                    )

            else:
                no_improve_count += 1

            current_1_mae = 1.0 - val_avg_loss
            seed_best_1_mae = 1.0 - seed_best_mae

            logger.debug(
                'Seed: {} , Epoch: {} , '
                '1-MAE: {:.4f} , seed best 1-MAE: {:.4f} , '
                'patience: {}/{} , '
                '1-Training Loss: {:.4f} , '
                '1-val_avg_o_loss: {:.4f} , '
                '1-val_avg_c_loss: {:.4f} , '
                '1-val_avg_e_loss: {:.4f} , '
                '1-val_avg_a_loss: {:.4f} , '
                '1-val_avg_n_loss: {:.4f}'.format(
                    seed,
                    epoch + 1,
                    current_1_mae,
                    seed_best_1_mae,
                    no_improve_count,
                    patience,
                    1.0 - train_avg_loss,
                    1.0 - val_avg_o_loss,
                    1.0 - val_avg_c_loss,
                    1.0 - val_avg_e_loss,
                    1.0 - val_avg_a_loss,
                    1.0 - val_avg_n_loss
                )
            )

            if no_improve_count >= patience:

                logger.info(
                    'Early stopping: seed={} at epoch {}. '
                    'Best epoch={}, best MAE={:.6f}, '
                    'best 1-MAE={:.6f}'.format(
                        seed,
                        epoch + 1,
                        seed_best_epoch,
                        seed_best_mae,
                        1.0 - seed_best_mae
                    )
                )

                break

            torch.cuda.empty_cache()

        # ----------------------------------------------------
        # Store only scalar results for this seed
        # ----------------------------------------------------
        seed_results.append(
            {
                'seed': seed,
                'best_epoch': seed_best_epoch,
                'best_mae': float(seed_best_mae),
                'best_1_mae': float(1.0 - seed_best_mae)
            }
        )

        logger.info(
            'Seed {} finished: best_epoch={}, '
            'best MAE={:.6f}, best 1-MAE={:.6f}'.format(
                seed,
                seed_best_epoch,
                seed_best_mae,
                1.0 - seed_best_mae
            )
        )

        # ----------------------------------------------------
        # Release GPU / CPU references before next seed
        # ----------------------------------------------------
        del model
        del optimizer
        del train_dataloader
        del val_dataloader

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()


# ============================================================
# Final multi-seed statistics
# ============================================================
best_1_mae_scores = np.array(
    [result['best_1_mae'] for result in seed_results],
    dtype=np.float64
)

best_mae_scores = np.array(
    [result['best_mae'] for result in seed_results],
    dtype=np.float64
)

mean_1_mae = best_1_mae_scores.mean()
std_1_mae = best_1_mae_scores.std(ddof=1)

mean_mae = best_mae_scores.mean()
std_mae = best_mae_scores.std(ddof=1)

print('\n' + '=' * 70)
print('MULTI-SEED RESULTS')
print('=' * 70)

for result in seed_results:
    print(
        'Seed {} | Best Epoch {:3d} | '
        'MAE {:.6f} | 1-MAE {:.6f}'.format(
            result['seed'],
            result['best_epoch'],
            result['best_mae'],
            result['best_1_mae']
        )
    )

print('-' * 70)
print(
    'MAE    : {:.6f} ± {:.6f}'.format(
        mean_mae,
        std_mae
    )
)
print(
    '1-MAE  : {:.6f} ± {:.6f}'.format(
        mean_1_mae,
        std_1_mae
    )
)
print(
    'Global best checkpoint: seed={}, epoch={}, '
    'MAE={:.6f}, 1-MAE={:.6f}'.format(
        global_best_seed,
        global_best_epoch,
        global_best_mae,
        1.0 - global_best_mae
    )
)
print(
    'Saved checkpoint: {}'.format(
        best_model_path
    )
)
print('=' * 70)


[2026-09-20 21:19:48,875]::3903913711::INFO::================ Seed 1/5: 42 ================


---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=257


Seed 42 Val Epoch 1: 100%|██████████| 337/337 [00:30<00:00, 11.03it/s]
[2026-09-20 21:24:59,017]::3903913711::INFO::Global best checkpoint updated: seed=42, epoch=1, MAE=0.125908, 1-MAE=0.874092
[2026-09-20 21:24:59,019]::3903913711::DEBUG::Seed: 42 , Epoch: 1 , 1-MAE: 0.8741 , seed best 1-MAE: 0.8741 , patience: 0/20 , 1-Training Loss: 0.8661 , 1-val_avg_o_loss: 0.9101 , 1-val_avg_c_loss: 0.9121 , 1-val_avg_e_loss: 0.7473 , 1-val_avg_a_loss: 0.8851 , 1-val_avg_n_loss: 0.9158
Seed 42 Val Epoch 2: 100%|██████████| 337/337 [00:29<00:00, 11.27it/s]
[2026-09-20 21:30:08,232]::3903913711::INFO::Global best checkpoint updated: seed=42, epoch=2, MAE=0.097297, 1-MAE=0.902703
[2026-09-20 21:30:08,233]::3903913711::DEBUG::Seed: 42 , Epoch: 2 , 1-MAE: 0.9027 , seed best 1-MAE: 0.9027 , patience: 0/20 , 1-Training Loss: 0.9057 , 1-val_avg_o_loss: 0.9059 , 1-val_avg_c_loss: 0.8896 , 1-val_avg_e_loss: 0.8969 , 1-val_avg_a_loss: 0.9325 , 1-val_avg_n_loss: 0.8887
Seed 42 Val Epoch 3: 100%|██████████| 

---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=257


Seed 123 Val Epoch 1: 100%|██████████| 337/337 [00:30<00:00, 11.20it/s]
[2026-09-21 00:39:19,864]::3903913711::DEBUG::Seed: 123 , Epoch: 1 , 1-MAE: 0.8790 , seed best 1-MAE: 0.8790 , patience: 0/20 , 1-Training Loss: 0.8632 , 1-val_avg_o_loss: 0.9043 , 1-val_avg_c_loss: 0.9122 , 1-val_avg_e_loss: 0.8570 , 1-val_avg_a_loss: 0.8492 , 1-val_avg_n_loss: 0.8722
Seed 123 Val Epoch 2: 100%|██████████| 337/337 [00:30<00:00, 11.15it/s]
[2026-09-21 00:44:27,007]::3903913711::DEBUG::Seed: 123 , Epoch: 2 , 1-MAE: 0.9028 , seed best 1-MAE: 0.9028 , patience: 0/20 , 1-Training Loss: 0.9006 , 1-val_avg_o_loss: 0.8984 , 1-val_avg_c_loss: 0.8944 , 1-val_avg_e_loss: 0.8944 , 1-val_avg_a_loss: 0.9243 , 1-val_avg_n_loss: 0.9025
Seed 123 Val Epoch 3: 100%|██████████| 337/337 [00:30<00:00, 11.15it/s]
[2026-09-21 00:49:34,061]::3903913711::DEBUG::Seed: 123 , Epoch: 3 , 1-MAE: 0.9010 , seed best 1-MAE: 0.9028 , patience: 1/20 , 1-Training Loss: 0.9294 , 1-val_avg_o_loss: 0.9013 , 1-val_avg_c_loss: 0.9132 , 1-

...

---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=257


Seed 1004 Val Epoch 1: 100%|██████████| 337/337 [00:30<00:00, 11.08it/s]
[2026-09-21 09:08:53,160]::3903913711::DEBUG::Seed: 1004 , Epoch: 1 , 1-MAE: 0.8838 , seed best 1-MAE: 0.8838 , patience: 0/20 , 1-Training Loss: 0.8621 , 1-val_avg_o_loss: 0.9131 , 1-val_avg_c_loss: 0.9027 , 1-val_avg_e_loss: 0.8792 , 1-val_avg_a_loss: 0.8099 , 1-val_avg_n_loss: 0.9140
Seed 1004 Val Epoch 2: 100%|██████████| 337/337 [00:30<00:00, 11.13it/s]
[2026-09-21 09:14:01,719]::3903913711::DEBUG::Seed: 1004 , Epoch: 2 , 1-MAE: 0.8988 , seed best 1-MAE: 0.8988 , patience: 0/20 , 1-Training Loss: 0.9055 , 1-val_avg_o_loss: 0.9085 , 1-val_avg_c_loss: 0.9041 , 1-val_avg_e_loss: 0.8838 , 1-val_avg_a_loss: 0.8998 , 1-val_avg_n_loss: 0.8978
Seed 1004 Val Epoch 3: 100%|██████████| 337/337 [00:30<00:00, 10.93it/s]
[2026-09-21 09:19:10,932]::3903913711::DEBUG::Seed: 1004 , Epoch: 3 , 1-MAE: 0.9062 , seed best 1-MAE: 0.9062 , patience: 0/20 , 1-Training Loss: 0.9348 , 1-val_avg_o_loss: 0.8988 , 1-val_avg_c_loss: 0.905

---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=257


Seed 2024 Val Epoch 1: 100%|██████████| 337/337 [00:30<00:00, 10.97it/s]
[2026-09-21 12:40:01,900]::3903913711::DEBUG::Seed: 2024 , Epoch: 1 , 1-MAE: 0.8967 , seed best 1-MAE: 0.8967 , patience: 0/20 , 1-Training Loss: 0.8649 , 1-val_avg_o_loss: 0.9079 , 1-val_avg_c_loss: 0.9025 , 1-val_avg_e_loss: 0.8820 , 1-val_avg_a_loss: 0.8839 , 1-val_avg_n_loss: 0.9070
Seed 2024 Val Epoch 2: 100%|██████████| 337/337 [00:30<00:00, 10.92it/s]
[2026-09-21 12:45:11,434]::3903913711::DEBUG::Seed: 2024 , Epoch: 2 , 1-MAE: 0.8780 , seed best 1-MAE: 0.8967 , patience: 1/20 , 1-Training Loss: 0.9065 , 1-val_avg_o_loss: 0.8607 , 1-val_avg_c_loss: 0.8832 , 1-val_avg_e_loss: 0.8926 , 1-val_avg_a_loss: 0.8810 , 1-val_avg_n_loss: 0.8724
Seed 2024 Val Epoch 3: 100%|██████████| 337/337 [00:30<00:00, 11.12it/s]
[2026-09-21 12:50:20,325]::3903913711::DEBUG::Seed: 2024 , Epoch: 3 , 1-MAE: 0.8907 , seed best 1-MAE: 0.8967 , patience: 2/20 , 1-Training Loss: 0.9306 , 1-val_avg_o_loss: 0.8689 , 1-val_avg_c_loss: 0.882


MULTI-SEED RESULTS
Seed 42 | Best Epoch  18 | MAE 0.089853 | 1-MAE 0.910147
Seed 123 | Best Epoch  25 | MAE 0.090328 | 1-MAE 0.909672
Seed 777 | Best Epoch  34 | MAE 0.089966 | 1-MAE 0.910034
Seed 1004 | Best Epoch  21 | MAE 0.093444 | 1-MAE 0.906556
Seed 2024 | Best Epoch  26 | MAE 0.093531 | 1-MAE 0.906469
----------------------------------------------------------------------
MAE    : 0.091424 ± 0.001892
1-MAE  : 0.908576 ± 0.001892
Global best checkpoint: seed=42, epoch=18, MAE=0.089853, 1-MAE=0.910147
Saved checkpoint: ./save_swintransformer_folder/model_best_multiseed.pth


In [13]:
# from torchinfo import summary

# summary(model, input_size = (4,15,4,3,224,224))